### **双固定效应回归（省份 FE + 年份 FE，省份聚类稳健 SE）**
分三层:
1. 模型在做什么（你在论文里怎么说）
2. 完整、可直接运行的 Python 代码（重点）
3. 结果怎么解读 + 下一步你该画什么图

### **一、标准的双向固定效应面板模型：**

  ***AMR_AGG_z_i,t ​= βX_i,t ​+ μ_i ​+ λ_t ​+ ε_i,t​***

- 𝑖：省份
- 𝑡：年份
- 𝜇_𝑖：省份固定效应（控制所有不随时间变化的省份特征）
- 𝜆_𝑡：年份固定效应（控制全国层面的时间冲击）
- 标准误：按省份聚类（cluster by province）

### **二、完整可运行代码**

In [4]:
# 导入包
import pandas as pd
import numpy as np

from linearmodels.panel import PanelOLS
import statsmodels.api as sm


In [9]:
# 读取数据
amr_path = r"C:\Users\lunch\Downloads\amr_rate.csv"
climate_path = r"C:\Users\lunch\Downloads\climate_social_eco.csv"

df_amr = pd.read_csv(amr_path)
df_x = pd.read_csv(climate_path)

def rename_key(df):
    m={}
    for c in df.columns:
        s = str(c).strip()
        if s.lower() in ["province","省份"]: m[c] = "Province"
        if s.lower() in ["year","年份"] or s == "YEAR": m[c] = "Year"
    return df.rename(columns=m)

df_amr = rename_key(df_amr)
df_x = rename_key(df_x)

print(df_amr.shape, df_x.shape)

(313, 15) (341, 31)


In [10]:
# 合并 AMR + 气候/污染/社会经济
## 这里假设 Province + Year 是共同键（前面就是这么做的？）

df = pd.merge(
    df_amr,
    df_x,
    on=["Province", "Year"],
    how="inner"
)

print("Merged shape:", df.shape)


Merged shape: (310, 44)


In [11]:
# 构造 AMR_AGG_z
amr_cols = [
    'MRCNS','VREFS','VREFM','PRSP','ERSP',
    '3GCRKP','MRSA','3GCREC','CREC',
    'QREC','CRPA','CRKP','CRAB'
]

amr_cols = [c for c in amr_cols if c in df.columns]

if "AMR_AGG_z" not in df.columns:
    amr_z = df[amr_cols].apply(
        lambda x: (x - x.mean()) / x.std(),
        axis=0
    )
    df["AMR_AGG_z"] = amr_z.mean(axis=1)
    print("✅ AMR_AGG_z created")
else:
    print("✅ AMR_AGG_z already exists")


✅ AMR_AGG_z created


In [12]:
# 选择指定的 9 个解释变量

X_vars = [
    "主要城市平均气温",
    "主要城市降水量",
    "R1xday",
    "PM2.5",
    "医疗水平",
    "GDP",
    "城市用水普及率",
    "生活垃圾无害化处理率",
    "抗菌药物使用强度"
]

# 确保列存在
X_vars = [c for c in X_vars if c in df.columns]
print("X used:", X_vars)


X used: ['主要城市平均气温', '主要城市降水量', 'R1xday', 'PM2.5', '医疗水平', 'GDP', '城市用水普及率', '生活垃圾无害化处理率', '抗菌药物使用强度']


In [13]:
# 构造 Panel 数据结构（关键一步）
df = df.set_index(["Province", "Year"])


In [14]:
# 准备回归数据（自动处理缺失）

Y = df["AMR_AGG_z"]
X = df[X_vars]

# 删除缺失
data = pd.concat([Y, X], axis=1).dropna()

Y = data["AMR_AGG_z"]
X = data[X_vars]

# PanelOLS 需要显式常数（但 FE 会吸收）
X = sm.add_constant(X)

print("Final N:", len(Y))


Final N: 239


In [15]:
# 8｜双固定效应回归（省份 FE + 年份 FE）

model = PanelOLS(
    Y,
    X,
    entity_effects=True,   # 省份 FE
    time_effects=True      # 年份 FE
)

res = model.fit(
    cov_type="clustered",
    cluster_entity=True   # 按省份聚类稳健SE
)

print(res.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:              AMR_AGG_z   R-squared:                        0.0341
Estimator:                   PanelOLS   R-squared (Between):              0.1155
No. Observations:                 239   R-squared (Within):              -0.5095
Date:                Tue, Feb 03 2026   R-squared (Overall):             -0.0315
Time:                        22:50:19   Log-likelihood                    85.114
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.7573
Entities:                          30   P-value                           0.6560
Avg Obs:                       7.9667   Distribution:                   F(9,193)
Min Obs:                       7.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             0.7046
                            

In [16]:
# 1️⃣ 提取回归结果（核心代码）
import pandas as pd
import numpy as np

# 从 PanelOLS 结果中提取
params = res.params
stderr = res.std_errors
tstat = res.tstats
pval = res.pvalues
ci = res.conf_int()

reg_table = pd.DataFrame({
    "Coefficient": params,
    "Std.Err": stderr,
    "t-stat": tstat,
    "P>|t|": pval,
    "CI_lower": ci.iloc[:, 0],
    "CI_upper": ci.iloc[:, 1]
})

# 去掉常数项（论文一般不放）
reg_table = reg_table.drop(index="const", errors="ignore")


In [17]:
# 2️⃣ 加显著性星号（规范版）
def signif_star(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.1:
        return "*"
    else:
        return ""

reg_table["Signif"] = reg_table["P>|t|"].apply(signif_star)

# 生成“系数（标准误）”格式
reg_table["Coef (SE)"] = (
    reg_table["Coefficient"].round(4).astype(str)
    + reg_table["Signif"]
    + "\n("
    + reg_table["Std.Err"].round(4).astype(str)
    + ")"
)


In [18]:
# 3️⃣ 重命名变量（中文论文直接用）
var_rename = {
    "主要城市平均气温": "气温",
    "主要城市降水量": "降水量",
    "R1xday": "极端降水（R1xday）",
    "PM2.5": "PM2.5",
    "医疗水平": "医疗水平",
    "GDP": "GDP",
    "城市用水普及率": "城市用水普及率",
    "生活垃圾无害化处理率": "生活垃圾无害化处理率",
    "抗菌药物使用强度": "抗菌药物使用强度"
}

reg_table.index = reg_table.index.map(lambda x: var_rename.get(x, x))


In [21]:
# ✅ 1）CSV（可追溯）
reg_table.to_csv("AMR_AGG_z_FE_results.csv", encoding="utf-8-sig")

# ✅ 2）Excel
with pd.ExcelWriter("AMR_AGG_z_FE_results.xlsx", engine="xlsxwriter") as writer:
    reg_table.to_excel(writer, sheet_name="FE_results")

# ✅ 3）LaTeX（直接进论文）
latex_table = reg_table[["Coef (SE)", "t-stat", "P>|t|"]].to_latex(
    caption="Two-way fixed effects regression of aggregate AMR (z-score)",
    label="tab:amr_fe",
    float_format="%.4f"
)

with open("AMR_AGG_z_FE_results.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)
